In [ ]:
import pandas as pd
import sqlite3
import os

# Paths (adjust if needed)
data_path = '../raw_data/'
cust_file = 'customer_data.csv'
trans_file = 'transaction_data.csv'
bank_file = 'bank_data.csv'

# Load CSVs
df_customers = pd.read_csv(os.path.join(data_path, cust_file))
df_transactions = pd.read_csv(os.path.join(data_path, trans_file))
df_bank = pd.read_csv(os.path.join(data_path, bank_file))

# Quick exploration
print("Customers shape:", df_customers.shape)
print(df_customers.head(3))
print("\nTransactions shape:", df_transactions.shape)
print(df_transactions.head(3))
print("\nBank/Branch shape:", df_bank.shape)
print(df_bank.head(3))

# Basic cleaning example (adapt to your data)
df_customers = df_customers.drop_duplicates(subset=['Customer_ID'])
df_transactions['Transaction_Date'] = pd.to_datetime(df_transactions['Transaction_Date'], errors='coerce')  # adjust column name
df_transactions['Transaction_Amount'] = pd.to_numeric(df_transactions['Transaction_Amount'], errors='coerce')

# Create SQLite DB for SQL practice (very useful for job demo)
conn = sqlite3.connect('banking_dashboard.db')

df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_transactions.to_sql('transactions', conn, if_exists='replace', index=False)
df_bank.to_sql('branches', conn, if_exists='replace', index=False)

# Example SQL queries – run these!
query1 = """
SELECT
    c.Region,
    COUNT(DISTINCT c.Customer_ID) AS num_customers,
    SUM(t.Transaction_Amount) AS total_transaction_volume,
    AVG(t.Transaction_Amount) AS avg_transaction_amount
FROM customers c
JOIN transactions t ON c.Customer_ID = t.Customer_ID
GROUP BY c.Region
ORDER BY total_transaction_volume DESC
LIMIT 10;
"""

top_regions = pd.read_sql_query(query1, conn)
print(top_regions)

# Another one: Customer age groups & investment behavior
query2 = """
SELECT
    CASE
        WHEN Age < 30 THEN '18-29'
        WHEN Age < 45 THEN '30-44'
        WHEN Age < 60 THEN '45-59'
        ELSE '60+'
    END AS age_group,
    COUNT(*) AS num_customers,
    AVG(Investment_Amount) AS avg_investment  -- adjust column if named differently
FROM customers
GROUP BY age_group
ORDER BY avg_investment DESC;
"""

age_invest = pd.read_sql_query(query2, conn)
print(age_invest)

conn.close()